# Step 4 runner: pull repo and run UPR-CRE v0.1

This notebook **does not edit source code**. It only pulls the `feature/upr-cre` branch, checks that Step 4 files/markers exist, prepares RegDB, and runs UPR-CRE v0.1.

In [13]:
CFG = {
    "GITHUB_USERNAME": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",  # empty = auto-detect by prepare_regdb_kaggle.py
    "USE_GITHUB_TOKEN": False,  # repo is public now; set True only if you make it private again
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "RUN_INSTALL": True,
    "RUN_DATASET_CHECK": True,
    "RUN_DIAGNOSTIC": True,
    "RUN_NAME": "relation_diag_regdb_smoke",
    "DEVICE": "0",
    "SEED": "1",
    # 4v0.1
    # "RUN_SMOKE": True,
    "RUN_SMOKE": False,
    "RUN_DEV": True,
    "RUN_ABLATION": True,
    "RUN_ABLATION_SMOKE": True
}
CFG


{'GITHUB_USERNAME': 'TranTruongMMCII',
 'REPO_NAME': 'UIT.CS2309',
 'BRANCH': 'feature/upr-cre',
 'WORK_DIR': '/kaggle/working',
 'DATA_ROOT': '/kaggle/working/VIREID_Dataset',
 'REGDB_SOURCE': '/kaggle/input/datasets/xqq027/reg-db/RegDB',
 'USE_GITHUB_TOKEN': False,
 'GITHUB_TOKEN_SECRET': 'GITHUB_TOKEN',
 'RUN_INSTALL': True,
 'RUN_DATASET_CHECK': True,
 'RUN_DIAGNOSTIC': True,
 'RUN_NAME': 'relation_diag_regdb_smoke',
 'DEVICE': '0',
 'SEED': '1',
 'RUN_SMOKE': False,
 'RUN_DEV': True,
 'RUN_ABLATION': True,
 'RUN_ABLATION_SMOKE': True}

In [14]:
from pathlib import Path
import os
import subprocess

cfg = CFG
work_dir = Path(cfg["WORK_DIR"])
repo_dir = work_dir / cfg["REPO_NAME"]
repo_url = f"https://github.com/{cfg['GITHUB_USERNAME']}/{cfg['REPO_NAME']}.git"

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", cfg["BRANCH"], repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", cfg["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", cfg["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{cfg['BRANCH']}"], cwd=repo_dir, check=True)

print("Repo:", repo_dir)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=True)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Your branch is behind 'origin/feature/upr-cre' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 64f0adf Change output flag to csv-output in script
Repo: /kaggle/working/UIT.CS2309
64f0adf


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
   4a28007..64f0adf  feature/upr-cre -> origin/feature/upr-cre
Already on 'feature/upr-cre'


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [15]:
from pathlib import Path

wsl_dir = repo_dir / "WSL_ReID"
required_files = [
    "main.py",
    "wsl.py",
    "task/train.py",
    "relation_metrics.py",
    "requirements-kaggle.txt",
    "scripts/apply_kaggle_compat_patches.py",
    "scripts/prepare_regdb_kaggle.py",
    "scripts/check_kaggle_env.py",
    "scripts/collect_relation_stats.py",
    "scripts/run_regdb_upr_v01.sh",
    "scripts/run_regdb_upr_v01_ablation.sh",
]
missing = [p for p in required_files if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing Step 4 required files: " + ", ".join(missing))

checks = {
    "main.py": ["--upr-cre", "--upr-beta", "--upr-gamma", "--upr-margin-weight", "--upr-warmup-epoch"],
    # Current canonical helper names in your repo:
    "wsl.py": ["_apply_upr_cre_score", "_prototype_score_for_rows", "_row_confidence", "last_rgb_score_refined", "dists = self._apply_upr_cre_score"],
    "scripts/prepare_regdb_kaggle.py": ["--data-root", "--regdb-source"],
    "scripts/run_regdb_upr_v01.sh": ["--upr-cre", "--upr-beta", "--upr-gamma", "--data-root", "--regdb-source", '"${CMD[@]}"'],
}
for rel, markers in checks.items():
    text = (wsl_dir / rel).read_text()
    for marker in markers:
        if marker not in text:
            raise RuntimeError(f"Missing marker {marker!r} in {rel}")

# Prevent old Step 3/Step 4 argument names from coming back.
script_text = (wsl_dir / "scripts/run_regdb_upr_v01.sh").read_text()
for bad in ["--source", "--output", "--input-root"]:
    if bad in script_text:
        raise RuntimeError(f"Old RegDB prepare argument {bad!r} found in run_regdb_upr_v01.sh")

print("Step 4 file/marker check OK.")


Step 4 file/marker check OK.


In [16]:
import subprocess

subprocess.run(["python", "-m", "py_compile", "main.py", "wsl.py", "relation_metrics.py", "scripts/collect_relation_stats.py"], cwd=wsl_dir, check=True)
subprocess.run(["bash", "-n", "scripts/run_regdb_upr_v01.sh"], cwd=wsl_dir, check=True)
subprocess.run(["bash", "-n", "scripts/run_regdb_upr_v01_ablation.sh"], cwd=wsl_dir, check=True)
print("Static py_compile and bash -n checks OK.")


Static py_compile and bash -n checks OK.


In [17]:
import subprocess, os

subprocess.run(["pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=wsl_dir, check=True)
subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)

env = os.environ.copy()
env["DATA_ROOT"] = CFG["DATA_ROOT"]
if CFG["REGDB_SOURCE"]:
    env["REGDB_SOURCE"] = CFG["REGDB_SOURCE"]

if CFG["RUN_DATASET_CHECK"]:
    prep = ["python", "scripts/prepare_regdb_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
    if CFG["REGDB_SOURCE"]:
        prep += ["--regdb-source", CFG["REGDB_SOURCE"]]
    subprocess.run(prep, cwd=wsl_dir, check=True, env=env)
    subprocess.run(["python", "scripts/check_kaggle_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True, env=env)


No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Thermal/285/male_back_t_05528_285.bmp
exists: True
image size: (64, 128

In [18]:
import subprocess, os

env = os.environ.copy()
env["DATA_ROOT"] = CFG["DATA_ROOT"]
if CFG["REGDB_SOURCE"]:
    env["REGDB_SOURCE"] = CFG["REGDB_SOURCE"]

if CFG["RUN_SMOKE"]:
    env.update({
        "RUN_NAME": "upr_v01_regdb_smoke_runner_v4",
        "STAGE1_EPOCH": "1",
        "STAGE2_EPOCH": "2",
        "MILESTONES": "1 2",
        "BATCH_PIDNUM": "2",
        "PID_NUMSAMPLE": "2",
        "TEST_BATCH": "32",
        "NUM_WORKERS": "2",
        "ENABLE_UPR_CRE": "1",
        "SAVE_RELATION_STATS": "1",
        "UPR_BETA": "0.2",
        "UPR_GAMMA": "0.5",
        "UPR_WARMUP_EPOCH": "1",
    })
    subprocess.run(["bash", "scripts/run_regdb_upr_v01.sh"], cwd=wsl_dir, check=True, env=env)

if CFG["RUN_DEV"]:
    env.update({
        "RUN_NAME": "upr_v01_regdb_s5_s15_runner_v4",
        "STAGE1_EPOCH": "5",
        "STAGE2_EPOCH": "15",
        "MILESTONES": "8 12",
        "BATCH_PIDNUM": "5",
        "PID_NUMSAMPLE": "4",
        "TEST_BATCH": "64",
        "NUM_WORKERS": "2",
        "ENABLE_UPR_CRE": "1",
        "SAVE_RELATION_STATS": "1",
        "UPR_BETA": "0.2",
        "UPR_GAMMA": "0.5",
        "UPR_WARMUP_EPOCH": "1",
    })
    subprocess.run(["bash", "scripts/run_regdb_upr_v01.sh"], cwd=wsl_dir, check=True, env=env)

if CFG["RUN_ABLATION_SMOKE"]:
    env.update({
        "BASE_RUN_PREFIX": "step4_v01_ablation_smoke_runner_v4",
        "STAGE1_EPOCH": "1",
        "STAGE2_EPOCH": "2",
        "MILESTONES": "1 2",
        "SAVE_RELATION_STATS": "1",
    })
    subprocess.run(["bash", "scripts/run_regdb_upr_v01_ablation.sh"], cwd=wsl_dir, check=True, env=env)


[UPR-CRE v0.1] repo: /kaggle/working/UIT.CS2309/WSL_ReID
[UPR-CRE v0.1] run: upr_v01_regdb_s5_s15_runner_v4
[UPR-CRE v0.1] data root: /kaggle/working/VIREID_Dataset
[UPR-CRE v0.1] RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
[UPR-CRE v0.1] device: 0
[UPR-CRE v0.1] enable_upr_cre: 1
64f0adf
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male

Traceback (most recent call last):
  File "/kaggle/working/UIT.CS2309/WSL_ReID/main.py", line 191, in <module>
    main(args)
  File "/kaggle/working/UIT.CS2309/WSL_ReID/main.py", line 67, in main
    _, result = train(args, model, dataset, current_epoch,cma,logger,enable_phase1)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/UIT.CS2309/WSL_ReID/task/train.py", line 282, in train
    model.optimizer_phase2.step()
  File "/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py", line 166, in wrapper
    return func.__get__(opt, opt.__class__)(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/optim/optimizer.py", line 526, in wrapper
    out = func(*args, **kwargs)
          ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/optim/optimizer.py", line 81, in _use_grad
    ret = func(*args, **kwargs)
          ^^

KeyboardInterrupt: 

In [20]:
from pathlib import Path
import pandas as pd

base = repo_dir / "saved_regdb_resnet"
if not base.exists():
    base = repo_dir / ".." / "saved_regdb_resnet"

summaries = sorted(base.glob("**/relation_stats_summary.csv")) if base.exists() else []
print("Found summaries:", len(summaries))
for p in summaries[-10:]:
    print(p)

if summaries:
    display(pd.read_csv(summaries[-1]))


Found summaries: 0
